## Introduction

Equity factors are specific, measurable characteristics of stocks—such as value, quality, momentum, size, volatility, and growth—that explain risk and return drivers beyond traditional market indexing. This project constructs five style factors from raw market data, stores them in a PostgreSQL database, and rigorously evaluates their predictive power using standard quantitative research methodology.

The analysis covers the S&P 500 universe using daily price data from 2021 to 2026 (~500 tickers, sourced via yfinance) and quarterly fundamental data from mid-2024 onward (income statements and balance sheets). Factor returns are benchmarked against published Fama-French factors from the Kenneth French Data Library. Key limitations include survivorship bias (only current index constituents are used), a short fundamental data window (~5-6 quarters), and a large-cap-only universe where some factor premiums are known to be weaker.

## Data & Methodology

### Universe and Data Sources

The investment universe consists of S&P 500 constituents as of the analysis date. Daily price data (open, high, low, adjusted close, volume) was downloaded via yfinance and stored in a PostgreSQL database. Quarterly fundamental data (net income, total revenue, stockholders' equity, total debt, ordinary shares outstanding) was sourced from yfinance financial statements. Fama-French daily factor returns (MKT-RF, SMB, HML, UMD, RF) were downloaded from the Kenneth French Data Library and compounded to monthly frequency.

SQL views transform the raw data into analytical building blocks: monthly returns (last trading day close-to-close), forward returns (next month's return, used to avoid look-ahead bias), and derived fundamentals (ROE, debt-to-equity, and market capitalization computed from raw inputs). All factor evaluations use forward returns — stocks are scored at the end of month T and evaluated on their performance in month T+1.

### Factor Definitions

Five factors are constructed from the raw data:

- **Momentum (12-1):** Cumulative compounded return over months T-12 to T-2, skipping the most recent month to avoid short-term reversal effects. Higher past returns signal continued outperformance.
- **Value:** Book-to-market ratio (stockholders' equity / market capitalization). Higher values indicate cheaper stocks relative to their accounting value.
- **Size:** Negated market capitalization. Smaller companies receive higher scores, reflecting the academic finding that small-cap stocks tend to outperform.
- **Volatility:** Negated 252-day rolling standard deviation of daily returns. Lower volatility receives higher scores, reflecting the low-volatility anomaly.
- **Quality:** Return on equity (net income / stockholders' equity). Higher profitability signals a stronger business.

Each factor's raw scores are z-score normalized cross-sectionally (across all stocks on a given date) and assigned to quintiles (1 = lowest score, 5 = highest). Price-based factors (momentum, volatility) use the full ~5-year price history. Fundamental-based factors (value, size, quality) are limited to ~5-6 quarters of data, with quarterly values forward-filled to monthly frequency using the most recent available report.

### Evaluation Methodology

Factor performance is assessed through four lenses:

**Information Coefficient (IC):** The Spearman rank correlation between factor scores at time T and stock returns at time T+1, computed monthly. Mean IC, standard deviation, information ratio (mean IC / std), and hit rate (percentage of months with positive IC) summarize each factor's predictive consistency.

**Quintile Spread Analysis:** Each month, stocks are sorted into five quintiles by factor score. The average forward return per quintile is computed across all months. A monotonic increase from Q1 to Q5 indicates the factor differentiates returns effectively.

**Long-Short Backtest:** A portfolio goes long the top quintile (Q5) and short the bottom quintile (Q1), rebalanced monthly. Cumulative returns, annualized Sharpe ratio, and maximum drawdown are computed. Portfolio turnover is measured and transaction costs are modeled at 10 basis points per trade.

**Statistical Significance:** Mean long-short returns are tested against zero using OLS regression with Newey-West (HAC) standard errors (6 lags) to correct for autocorrelation in monthly return series. P-values are adjusted for multiple hypothesis testing using both Bonferroni correction and Benjamini-Hochberg false discovery rate control.

### Bias Awareness

- **Survivorship bias:** The universe uses current S&P 500 constituents. Companies that were delisted or removed from the index are absent, which may overstate factor returns. Point-in-time constituent data would address this but is not freely available.
- **Look-ahead bias:** All factor scores use only data available at the time of scoring. Fundamental data is aligned using the most recent quarterly report on or before the scoring date, and factor performance is measured using forward (next-month) returns.

## Results

In [2]:
import sys
sys.path.insert(0, 'C:/Users/pale4/projects/factor-research-platform')

from analysis.correlation_analysis import compute_factor_correlations, plot_correlation_heatmap

## Conclusions